# UFTC B-747 — координированный разворот с отказом двигателя в середине манёвра

Этот ноутбук — **манёвренный** аналог демо Phase 1 / Phase 3 / Phase 4 на стационарном трим-режиме. Вместо удержания прямолинейного крейсерского полёта самолёт выполняет 300-секундный сценарий: координированный разворот с креном, после которого следует длительный участок удержания нового курса:

| Окно (с) | Этап | Что меняется |
|----------|------|--------------|
| 0 – 5    | горизонтальный крейс | удержание V, h, ψ = 0 |
| 5 – 15   | вход в крен | φ нарастает 0 → +15° (правый крен) |
| 15 – 127.95 | установившийся разворот | φ = +15°, ψ меняется ≈0.73°/с (согласовано с `g·tan(φ)/V`) |
| 127.95 – 137.95 | выход из крена | φ снижается +15° → 0 |
| 137.95 – 300 | удержание курса | φ = 0, ψ = ψ_final |

**Отказ**: **левый внешний двигатель отказывает на t = 50 с**, *в середине разворота*, так что UFTC должен продолжать отслеживание движущегося задания при асимметричной тяге.

Сравниваем две конфигурации:
* **Phase 1** — внутренний контур L2 AA-INDI + средний L3 IADP + envelope-аллокатор, **без внешнего L4 и без монитора**.
* **Phase 4** — полный стек UFTC: предобученный внешний L4 D-SAC + композитный Ляпуновский монитор + диспетчер макро-действий.

**Plant-agnostic заметка** — задание UFTC `reference` всё время равно `0`. Время-зависимый манёвр заводится в *уставку envelope-аллокатора* `ref0(t)`, которую `uftc_state_transform` превращает в state в форме ошибки. UFTC видит нетривиальную дрейфующую ошибку, и каскад компенсирует её онлайн.


## 1. Импорты и параметры симуляции


In [ ]:
import warnings                                            # стандартный модуль предупреждений
warnings.filterwarnings("ignore")                            # подавляем шумные warnings NumPy/Torch на время демо

import math                                                  # скалярная математика (radians, tan, sqrt, ...)
from pathlib import Path                                     # кроссплатформенная работа с путями
from collections import Counter                              # подсчёт типов сработавших макро-действий

import matplotlib.pyplot as plt                              # построение time-history графиков
import numpy as np                                           # массивы и линейная алгебра
from scipy.optimize import fsolve                            # численный решатель для трим-режима с отказом

from tensoraerospace.aerospacemodel.b747.nonlinear import B747Configuration, trim   # номинальная конфигурация + триммер
from tensoraerospace.aerospacemodel.b747.nonlinear.damage.state import B747DamageState  # полный вектор состояния отказов
from tensoraerospace.aerospacemodel.b747.nonlinear.dynamics import b747_ode_6dof    # 6-DoF ODE для невязки трим-системы
from tensoraerospace.aerospacemodel.b747.nonlinear.damage import (
    DamageProfile, EngineFailureEvent,                       # программируемый профиль отказов + событие отказа двигателя
)
from tensoraerospace.aerospacemodel.b747.nonlinear.engine import JT9DEngine          # модель тяги JT9D (на связку)
from tensoraerospace.aerospacemodel.b747.nonlinear.params import (
    default_parameters, isa_speed_of_sound_ft_s,             # номинальные параметры + хелпер скорости звука по ISA
)
import importlib                                             # понадобится для перезагрузки модуля контроллера
import tensoraerospace.agent.uftc.controller as uftc_controller_module  # верхнеуровневый оркестратор UFTC
importlib.reload(uftc_controller_module)                     # подхватываем локальные правки без рестарта ядра
from tensoraerospace.agent.uftc.controller import UFTCConfig, UFTCController  # датакласс конфига + сам контроллер
from tensoraerospace.agent.aa_indi.model import AAINDIConfig                  # конфиг внутреннего контура L2
from tensoraerospace.agent.iadp.model import IADPConfig                       # конфиг среднего контура L3
from tensoraerospace.agent.uftc.fdd.detector import FDDConfig                 # конфиг детектора отказов FDD/Kalman
from tensoraerospace.envs.b747_nonlinear import NonlinearB747Env              # gym-подобная нелинейная среда B-747

np.random.seed(0)                                            # детерминизм генератора NumPy
import torch                                                 # PyTorch (используется в actor/critic L4 D-SAC)
torch.manual_seed(0)                                         # детерминизм генератора Torch (CPU и CUDA)

DT = 0.05                                                    # шаг симуляции в секундах (20 Гц)
TOTAL_TIME = 300.0                                           # длительность эпизода, секунды
DAMAGE_TIME = 50.0                                           # время отказа левого внешнего двигателя
N_EP = int(TOTAL_TIME / DT)         # 6000 шагов             # число шагов интегрирования за эпизод
DAMAGE_STEP = int(DAMAGE_TIME / DT)                          # индекс шага, на котором запускается отказ

V_REF_FT_S = 674.0                                           # крейсерская истинная скорость (ft/s) ≈ M0.7 на FL200
ALT_REF_FT = 20_000.0                                        # крейсерская высота (футы)
PSI_REF_DEG = 0.0                                            # начальный курс (deg) — север

# Расписание координированного разворота. Знаковое соглашение: положительный
# крен phi = правое крыло вниз (правый крен). Для координированного правого
# разворота скорость курса:  psi_dot = (g/V) * tan(phi),
# положительный крен -> положительная скорость курса.
TURN_BANK_DEG  = 15.0     # пиковый угол крена (правый разворот -> положительный)
TURN_BANK_SIGN = +1.0     # правый разворот
ROLL_IN_S   = (5.0, 15.0)                                    # окно, в котором phi нарастает 0 -> +15°
STEADY_S    = (15.0, 127.95)                                 # окно установившегося крена на пике phi
ROLL_OUT_S  = (127.95, 137.95)                               # окно, в котором phi возвращается к 0

print(f"Длительность эпизода : {TOTAL_TIME:.0f} с ({N_EP} шагов @ dt={DT}с)")  # проверка горизонта
print(f"Триггер отказа       : t = {DAMAGE_TIME:.0f} с (шаг {DAMAGE_STEP})")    # проверка шага отказа
print(f"Профиль разворота    : вход {ROLL_IN_S}, установ {STEADY_S}, выход {ROLL_OUT_S}")  # окна разворота
print(f"Пиковый крен         : {TURN_BANK_SIGN*TURN_BANK_DEG:+.1f}° (правый крен)")        # подписанный пик крена


## 2. Трим-режим в крейсе и при отказе двигателя

Тот же расчёт триминга, что и в Phase 4: здоровый крейсерский трим плюс стационарный трим с отказом левого внешнего двигателя, в который envelope-аллокатор плавно перетекает во время переходного процесса отказа. Коэффициент смеси — `engine_loss_estimate ∈ [0, 1]`, его задаёт severity-выход FDD.


In [ ]:
trim_result = trim(altitude_ft=ALT_REF_FT, V_ft_s=V_REF_FT_S,    # ищем здоровый крейсерский трим на FL200
                   config=B747Configuration.NOMINAL)                # номинальная аэродинамическая/двигательная конфиг
assert trim_result.converged                                        # ассерт на сходимость триммера

delta_e_trim_rad = float(trim_result.elevator_rad)                  # угол стабилизатора, удерживающий 1g крейс
throttle_trim = float(trim_result.throttle)                         # положение РУД (в [0,1]) на крейсе
theta_ref_deg = math.degrees(trim_result.theta_rad)                 # тангаж в градусах, для печати
healthy_trim_action = np.array([                                    # 4-канальный вектор управления для здорового трима
    delta_e_trim_rad, 0.0, 0.0, throttle_trim,                      # de, da=0, dr=0, РУД
], dtype=np.float64)

engine_params = default_parameters(B747Configuration.NOMINAL)       # номинальные параметры (масса, инерции, ...)
engine_model = JT9DEngine(                                          # модель тяги JT9D — для распределения тяги по двигателям
    n_engines=4,                                                    # у B-747 четыре двигателя
    sls_thrust_per_engine_lb=engine_params.engine_thrust_max_lb / 4.0,  # SLS-тяга равно делится по двигателям
    idle_frac=engine_params.engine_idle_frac,                       # доля минимальной тяги (idle)
    spool_tau_s=engine_params.engine_tau_s,                         # постоянная времени раскрутки (секунды)
)

engine_out_params = default_parameters(B747Configuration.NOMINAL)   # копия параметров под трим с отказом
engine_out_damage_state = B747DamageState.healthy()                 # стартуем со здорового состояния
engine_out_damage_state.engines_mu[1] = 0.0                         # обнуляем мультипликатив mu двигателя №1 (левый внешний)
engine_out_params.damage_state = engine_out_damage_state            # подключаем damage_state с отказом
throttle_engine_out_estimate = min(throttle_trim * 4.0 / 3.0, 1.0)  # грубая оценка: тягу 4 двигателей перераспределяем на 3

def engine_out_state_from_vars(alpha_rad, beta_rad, theta_rad):     # собираем 12-вектор state в связанной СК
    return np.array([                                               # замыкание для fsolve
        V_REF_FT_S * math.cos(alpha_rad) * math.cos(beta_rad),      # u — скорость по оси x
        V_REF_FT_S * math.sin(beta_rad),                            # v — скорость по оси y (скольжение)
        V_REF_FT_S * math.sin(alpha_rad) * math.cos(beta_rad),      # w — скорость по оси z
        0.0, 0.0, 0.0,                                              # p, q, r — угловые скорости (в триме нули)
        0.0, theta_rad, 0.0,                                        # phi=0, theta, psi=0 (горизонталь, кроме alpha)
        0.0, 0.0, -ALT_REF_FT,                                      # x_e=0, y_e=0, z_e=-высота (NED)
    ], dtype=np.float64)

def engine_out_trim_residual(z):                                    # невязка, корнь которой — трим с отказом
    alpha_rad, beta_rad, theta_rad, de_rad, da_rad, dr_rad, throttle_cmd = z  # распаковка 7 свободных переменных
    x = engine_out_state_from_vars(alpha_rad, beta_rad, theta_rad)  # реконструируем полный state
    u = np.array([de_rad, da_rad, dr_rad, np.clip(throttle_cmd, 0.0, 1.0)],   # вектор управления, РУД клипуем
                 dtype=np.float64)
    dx = b747_ode_6dof(x, u, 0.0, engine_out_params)                # 6-DoF ODE с активным damage-состоянием
    return np.array([dx[0], dx[1], dx[2], dx[3], dx[4], dx[5], dx[11]], dtype=np.float64)  # обнуляем u̇,v̇,ẇ,ṗ,q̇,ṙ,ḣ

z0 = np.array([trim_result.alpha_rad, math.radians(-0.44), trim_result.theta_rad,  # начальное приближение от здорового трима
               delta_e_trim_rad, math.radians(-4.4), math.radians(-2.64),          # плюс типичные смещения da/dr
               throttle_engine_out_estimate], dtype=np.float64)                    # плюс увеличенная начальная РУД
engine_out_solution, info, ier, msg = fsolve(                                      # решаем невязку = 0
    engine_out_trim_residual, z0, full_output=True, xtol=1e-10, maxfev=2_000,      # жёсткая толерантность, щедрый бюджет
)
engine_out_residual_norm = float(np.linalg.norm(info["fvec"]))                     # норма финальной невязки
assert ier == 1 and engine_out_residual_norm < 1e-6, msg                           # ассерт сходимости

(alpha_engine_out_rad, beta_engine_out_rad, theta_engine_out_rad,                  # распаковываем сошедшееся решение
 delta_e_engine_out_rad, delta_a_engine_out_rad, delta_r_engine_out_rad,
 throttle_engine_out) = engine_out_solution
throttle_engine_out = float(np.clip(throttle_engine_out, 0.0, 1.0))                # на всякий случай клипуем в [0,1]
engine_out_trim_action = np.array([                                                # 4-канальный трим-вектор при отказе
    delta_e_engine_out_rad, delta_a_engine_out_rad,                                # de, da
    delta_r_engine_out_rad, throttle_engine_out,                                   # dr, РУД
], dtype=np.float64)

print(f"Здоровый трим @ FL200, V={V_REF_FT_S:.0f} ft/s:")                          # печать здорового трима
print(f"  alpha = theta = {theta_ref_deg:+.3f}°")                                  # AoA ≈ тангаж в триме (gamma=0)
print(f"  delta_e_trim  = {math.degrees(delta_e_trim_rad):+.3f}°, throttle_trim = {throttle_trim:.4f}")
print(f"Норма невязки трима с отказом = {engine_out_residual_norm:.2e}")           # подтверждаем малость невязки


## 3. Расписание задания — координированный разворот

Строим `reference_schedule(t)`, возвращающую словарь `{V, h, theta, psi, phi, beta}`. Расписание крена — трапеция; расписание курса — аналитический интеграл координированной скорости разворота ψ̇ ≈ g·tan(|φ|) / V на том же окне.

Числа (выбраны под аккуратный, а не агрессивный манёвр):
* Крен: пик +15° (правый разворот).
* Угловая скорость крена: ±1.5°/с во время рамп.
* На установе ψ̇ ≈ g·tan(15°) / V ≈ 32.17·0.268 / 674 ≈ 0.0128 рад/с ≈ **0.732°/с**.
* Полный Δψ ≈ 3.62° (вход) + 0.733·112.95° (установ) + 3.62° (выход) ≈ **90.0°**.

θ получает небольшую добавку +0.3° на участке разворота для компенсации наклона вектора подъёмной силы на 15° крене (прокси «L·cos(φ)»). β-задание остаётся 0 до отказа и плавно перетекает к β из трим-режима с отказом после события.


In [ ]:
G_FT_S2 = 32.17                                              # ускорение свободного падения в ft/s^2

def _phi_ref_rad(t: float) -> float:                             # задание угла крена в момент t (радианы)
    if t < ROLL_IN_S[0]:                                         # до входа в крен: крылья горизонтальны
        return 0.0
    if t < ROLL_IN_S[1]:                                         # окно входа в крен: линейная рампа 0 -> пик
        frac = (t - ROLL_IN_S[0]) / (ROLL_IN_S[1] - ROLL_IN_S[0])
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) * frac
    if t < STEADY_S[1]:                                          # установившийся разворот: держим пиковый крен
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG)
    if t < ROLL_OUT_S[1]:                                        # выход из крена: линейная рампа пик -> 0
        frac = 1.0 - (t - ROLL_OUT_S[0]) / (ROLL_OUT_S[1] - ROLL_OUT_S[0])
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) * frac
    return 0.0                                                    # после выхода: снова горизонтально

def _psi_dot_coordinated(phi_rad: float, V_ft_s: float) -> float:  # координированная скорость курса
    # Координированный разворот: psi_dot = (g/V) * tan(phi).
    # Положительный крен (правое крыло вниз) -> положительная скорость курса (правый разворот).
    return G_FT_S2 * math.tan(phi_rad) / max(V_ft_s, 1.0)         # max() защищает от V близкого к нулю

def _phi_dot_ref_rad_s(t: float) -> float:                       # аналитическая d/dt от phi_ref (FF для элеронов)
    if ROLL_IN_S[0] <= t < ROLL_IN_S[1]:                         # вход в крен: положительный наклон
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) / (ROLL_IN_S[1] - ROLL_IN_S[0])
    if ROLL_OUT_S[0] <= t < ROLL_OUT_S[1]:                       # выход из крена: отрицательный наклон
        return -TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) / (ROLL_OUT_S[1] - ROLL_OUT_S[0])
    return 0.0                                                    # на горизонтальных и установившемся участках — ноль

# Предвычисляем psi_ref(t) трапецией от psi_dot(phi_ref(t), V_REF).
_t_grid = np.arange(0.0, TOTAL_TIME + DT, DT)                    # равномерная сетка времени для интеграла
_phi_grid = np.array([_phi_ref_rad(float(t)) for t in _t_grid])  # phi_ref на сетке
_psi_dot_grid = np.array([_psi_dot_coordinated(float(p), V_REF_FT_S) for p in _phi_grid])  # psi_dot на сетке
_psi_grid = np.zeros_like(_t_grid)                               # аккумулятор psi_ref(t)
for i in range(1, len(_t_grid)):                                 # трапециевидная квадратура
    _psi_grid[i] = _psi_grid[i-1] + 0.5 * DT * (_psi_dot_grid[i-1] + _psi_dot_grid[i])
PSI_FINAL_RAD = float(_psi_grid[int(ROLL_OUT_S[1] / DT)])        # финальный курс после выхода (далее держим его)

def _psi_ref_rad(t: float) -> float:                             # кусочное задание курса
    if t <= 0.0:                                                 # до t=0: курс 0
        return 0.0
    if t >= ROLL_OUT_S[1]:                                       # после выхода: держим финальный курс
        return PSI_FINAL_RAD
    return float(np.interp(t, _t_grid, _psi_grid))               # линейная интерполяция по сетке

def _theta_offset_rad(t: float) -> float:                        # небольшой подъём тангажа на участке крена
    # Малая компенсация тангажа/подъёмной силы, плавно следующая за заданием крена.
    # Это убирает ступеньку в задании theta во время здорового разворота.
    peak_phi = max(math.radians(abs(TURN_BANK_DEG)), 1e-9)        # защита от деления на ноль на вырожденном случае
    bank_fraction = abs(_phi_ref_rad(t)) / peak_phi              # текущая доля |phi_ref| от пика
    return math.radians(0.3) * bank_fraction ** 2                # квадратичная рампа -> +0.3° на полном крене

def reference_schedule(t: float, beta_target_rad: float = 0.0) -> dict:  # публичный API rollout-а
    phi_ref = _phi_ref_rad(t)                                    # кешируем phi_ref для повторного использования
    return {                                                     # полный словарь задания, потребляемый uftc_state_transform_dyn
        "V": V_REF_FT_S,                                         # уставка скорости (ft/s)
        "h": ALT_REF_FT,                                         # уставка высоты (ft)
        "theta": float(trim_result.theta_rad) + _theta_offset_rad(t),  # уставка тангажа (рад), с поправкой на крен
        "psi": _psi_ref_rad(t),                                  # уставка курса (рад)
        "phi": phi_ref,                                          # уставка крена (рад)
        "beta": beta_target_rad,                                 # уставка скольжения (рад), 0 в здоровой фазе
        "phi_dot": _phi_dot_ref_rad_s(t),                        # FF по скорости крена для канала элеронов
        "psi_dot": _psi_dot_coordinated(phi_ref, V_REF_FT_S),    # FF по скорости курса для канала руля направления
    }

# Визуализация задания
phi_ref_deg = np.degrees(_phi_grid)                              # phi_ref в градусах для графика
psi_ref_deg = np.degrees(_psi_grid)                              # psi_ref в градусах для графика
psi_dot_deg_s = np.degrees(_psi_dot_grid)                        # psi_dot в град/с для графика

fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)   # 3 строки на общей оси времени
axes[0].plot(_t_grid, phi_ref_deg, color='tab:blue', lw=1.4)     # phi_ref(t)
axes[0].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5, label='отказ двигателя')  # маркер отказа
axes[0].set_ylabel('phi_ref, °'); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(_t_grid, psi_ref_deg, color='tab:green', lw=1.4)    # psi_ref(t)
axes[1].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
axes[1].set_ylabel('psi_ref, °'); axes[1].grid(True, alpha=0.3)
axes[2].plot(_t_grid, psi_dot_deg_s, color='tab:purple', lw=1.4) # psi_dot_ref(t)
axes[2].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
axes[2].set_xlabel('время, с'); axes[2].set_ylabel('psi_dot, °/с'); axes[2].grid(True, alpha=0.3)
axes[0].set_title('Задание координированного разворота')
plt.tight_layout(); plt.show()                                   # отрисовываем график задания
print(f'Полное изменение курса Δψ по расписанию = {math.degrees(PSI_FINAL_RAD):+.2f}°')    # финальный Δψ
print(f'psi_dot на установившемся пиковом крене  = '                                       # проверка пиковой скорости
      f'{math.degrees(_psi_dot_coordinated(TURN_BANK_SIGN*math.radians(TURN_BANK_DEG), V_REF_FT_S)):+.3f}°/с')


## 4. Профиль отказа — отказ двигателя на t = 50 с

Готовый пресет `LEFT_OUTER_ENGINE_FAILURE` срабатывает на t = 10 с. Здесь же конструируем локальный `DamageProfile` с теми же параметрами `EngineFailureEvent`, но с `trigger_time=DAMAGE_TIME=50 с`.


In [ ]:
TURN_ENGINE_FAILURE = DamageProfile(                         # программируемый профиль отказов
    events=[                                                  # список событий отказа (здесь одно)
        EngineFailureEvent(                                   # отказ левого внешнего двигателя
            trigger_time=DAMAGE_TIME,                         # ровно на DAMAGE_TIME = 50 с
            engine_id=1,                                      # двигатель №1 = левый внешний (нумерация B-747)
            thrust_fraction=0.0,                              # доля оставшейся тяги = 0 (полный отказ)
            label='left_outer_engine_flameout_mid_turn',      # человекочитаемая метка для логов
        ),
    ],
)
print(f'Профиль отказа: {TURN_ENGINE_FAILURE.events[0]}')      # печатаем единственное событие для проверки


## 5. Контроллер UFTC и хелперы

Состояние UFTC остаётся `[eV, eh, e_theta, e_psi, e_phi, e_beta, p, q, r]` — ровно как в Phase 4. Единственное изменение: `uftc_state_transform` теперь принимает **изменяющуюся во времени** уставку `ref_t` вместо замороженного `ref0`. Всё дальше по цепочке остаётся прежним.


In [ ]:
UFTC_RESIDUAL_SCALE = np.array([                                  # масштаб residual-действия: насколько UFTC может «дотолкнуть» канал
    0.0,                                                          # de: residual нет (отдан state-feedback)
    math.radians(0.05),                                           # da: ±0.05° residual элеронов
    math.radians(0.05),                                           # dr: ±0.05° residual руля направления
    0.0,                                                          # РУД: residual нет
], dtype=np.float64)

UFTC_STATE_SCALE = np.array([                                     # масштабы нормализации UFTC-state
    25.0, 250.0,                                                  # eV (ft/s), eh (ft)
    math.radians(3.0), math.radians(8.0),                         # e_theta (рад), e_psi (рад)
    math.radians(8.0), math.radians(2.0),                         # e_phi (рад), e_beta (рад)
    1.0, 1.0, 1.0,                                                # угловые скорости p, q, r (рад/с)
], dtype=np.float64)
N_UFTC_STATE = int(UFTC_STATE_SCALE.size)                         # размерность state (=9)
UFTC_OMEGA_INDICES = [6, 7, 8]                                    # индексы угловых скоростей (для L2)
UFTC_REFERENCE = np.zeros(N_UFTC_STATE, dtype=np.float64)         # нулевая reference: манёвр заводится через ref_t

UFTC_F_INIT = np.eye(N_UFTC_STATE, dtype=np.float64) * 0.99       # warm-start F: почти единичная (лёгкое затухание)
UFTC_F_INIT[2, 7] = DT / UFTC_STATE_SCALE[2]                      # theta интегрирует q с учётом нормализации
UFTC_F_INIT[3, 8] = DT / UFTC_STATE_SCALE[3]                      # psi интегрирует r с учётом нормализации
UFTC_F_INIT[4, 6] = DT / UFTC_STATE_SCALE[4]                      # phi интегрирует p с учётом нормализации

UFTC_G_INIT = np.zeros((N_UFTC_STATE, 4), dtype=np.float64)       # warm-start B: нулевая основа
UFTC_G_INIT[0] = [0.0, 0.0, 0.0, 0.08]                            # eV откликается в основном на РУД
UFTC_G_INIT[1] = [-0.02, 0.0, 0.0, 0.03]                          # eh — на стабилизатор (-) и РУД (+)
UFTC_G_INIT[6] = [0.00, 0.30, 0.00, 0.00]                         # p — на элероны
UFTC_G_INIT[7] = [0.40, 0.00, 0.00, 0.10]                         # q — на стабилизатор, слабо на РУД
UFTC_G_INIT[8] = [0.00, 0.05, 0.30, 0.00]                         # r — в основном на руль, чуть на элероны
UFTC_G_INNER = UFTC_G_INIT[UFTC_OMEGA_INDICES].copy()             # внутренний срез (только rates) для AA-INDI

L4_ACTION_SCALE = 0.05                                            # масштаб действия L4 D-SAC (малый residual)
L4_TRIM_FREE_INDICES = {'V_idx': 0, 'gamma_idx': 1, 'alpha_idx': 2, 'q_idx': 7}  # trim-free индексы для L4

# Калибровка монитора Phase 4 — расслабленный d=(80,)*5, чтобы тревога оставалась
# содержательной во время длинного манёвра, который и так раскачивает V_INDI/V_FDD.
MONITOR_CFG = dict(
    enable_monitor=True,                                          # включаем композитный Ляпуновский монитор
    monitor_c_weights=(0.0, 0.4, 0.0, 0.3, 0.3),                  # веса (V_hj, V_indi, V_iadp, V_dsac, V_fdd)
    monitor_d_disturbance=(80.0,) * 5,                            # бюджет возмущений по компонентам
    monitor_alarm_warn_frac=0.4,                                  # WARN при 40% от mu_uub_pred
    monitor_alarm_critical_frac=0.7,                              # CRITICAL при 70% от mu_uub_pred
    monitor_cooldown_steps=20,                                    # кулдаун между срабатываниями макро-действий
)

def make_env(damage_profile=None, n_steps=N_EP + 5):              # создаём свежую нелинейную среду B-747
    return NonlinearB747Env(
        trim_at=(ALT_REF_FT, V_REF_FT_S),                         # высота и скорость трима для инициализации
        number_time_steps=n_steps,                                # длина эпизода
        dt=DT, integrator='rk4', action_space='virtual',          # 50 мс RK4, виртуальное управление
        config=B747Configuration.NOMINAL,                         # номинальная аэро/двигательная конфиг
        damage_profile=damage_profile,                            # опциональный профиль отказов
    )

def make_controller(*, enable_l4_outer=True, enable_trim_free=True,    # собираем UFTCController с заданными флагами
                    enable_monitor=True):
    cfg_kwargs = dict(
        dt=DT,                                                    # шаг управления (совпадает с env)
        fdd_warmup_steps=0,                                       # FDD активен с t=0
        omega_indices=UFTC_OMEGA_INDICES,                         # какие индексы state — угловые скорости
        middle_lookahead_dt=0.3,                                  # горизонт прогноза IADP (с)
        trust_radius_nominal=0.03,                                # доверительный радиус в номинальном режиме
        trust_radius_fault=0.15,                                  # ослабленный радиус после декларации отказа
        fdd_cfg=FDDConfig(process_noise=1e-4, measurement_noise=1e-3,
                          adapt_Q=False, adapt_R=False, drift=6.0, h_alarm=15.0),  # фильтр Калмана FDD + пороги тревоги
        inner_cfg=AAINDIConfig(dt=DT, ref_wn=3.0, ref_zeta=0.9,    # конфиг L2 AA-INDI
                               u_magnitude_limit=1.0, u_rate_limit=5.0,
                               G_init=UFTC_G_INNER, ref_error_kp=2.0,
                               ref_error_ki=0.05, seed=0),
        middle_cfg=IADPConfig(dt=DT,                                # конфиг L3 IADP
                              Q=np.diag([20., 40., 20., 20., 20., 0.5, 2., 2., 2.]),  # штраф state по каналам
                              R=np.diag([8., 8., 8., 20.]),                          # штраф управления по каналам
                              gamma=0.85, gamma_rls=0.995,                            # дисконт ADP + забывание RLS
                              u_magnitude_limit=1.0, u_rate_limit=4.0,                # лимиты амплитуды/скорости
                              policy_eval_every=80, policy_eval_blend=0.15),          # частота policy-eval + смешивание
        enable_l4_outer=bool(enable_l4_outer),                    # включить ли внешний L4 D-SAC
        l4_action_scale=L4_ACTION_SCALE,                          # масштаб действия L4
        l4_eval_mode=True,                                        # eval-режим: детерминированный актор, без exploration
        l4_seed=0,                                                # сид RNG L4
        l4_trim_free=(L4_TRIM_FREE_INDICES if enable_trim_free else None),  # каналы, где L4 может trim-free
    )
    if enable_monitor:                                            # опционально включаем композитный монитор
        cfg_kwargs.update(MONITOR_CFG)
    return UFTCController(                                        # финально собираем контроллер
        n_state=N_UFTC_STATE, n_control=4,                        # 9 state, 4 управления
        nominal_F=UFTC_F_INIT, nominal_G=UFTC_G_INIT,             # warm-start для идентификации
        config=UFTCConfig(**cfg_kwargs),                          # собранный конфиг
    )


In [ ]:
STATE_FEEDBACK_GAINS = {                                          # ручные PI/PD-коэффициенты для envelope-аллокатора
    'de_h': 2.5e-3,                                               # отклик стабилизатора на ошибку высоты
    'de_theta': 1.8,                                              # отклик стабилизатора на ошибку тангажа
    'de_q': 1.0,                                                  # демпфирование стабилизатором по q
    'throttle_v': -1.2e-1,                                        # отклик РУД на ошибку скорости
    'throttle_h': -2.0e-4,                                        # отклик РУД на ошибку высоты (малый bias)
    'da_phi': -2.5,                                               # отклик элеронов на ошибку крена
    'da_p': -1.5,                                                 # демпфирование элеронами по p
    'da_phi_dot': 2.25,                                           # FF элеронов от phi_dot_ref
    'dr_r': 3.0,                                                  # демпфирование рулём направления по r
    'dr_psi': 1.5,                                                # отклик руля направления на ошибку курса
    'dr_beta': -0.5,                                              # отклик руля направления на ошибку скольжения
    'dr_psi_dot': -3.0,                                           # FF руля направления от psi_dot_ref
}

def wrap_deg(angle_deg):                                          # привести угол в (-180, 180] в градусах
    return (float(angle_deg) + 180.0) % 360.0 - 180.0

def wrap_rad(angle_rad):                                          # привести угол в (-pi, pi] в радианах
    return (float(angle_rad) + math.pi) % (2.0 * math.pi) - math.pi

def true_airspeed_ft_s(obs):                                      # ||(u, v, w)||_2 в ft/s
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)           # «уплощаем» массив
    return float(np.linalg.norm(obs[:3]))                         # модуль вектора скорости в связанной СК

def altitude_ft(obs):                                             # высота в футах из NED z_e
    return float(-np.asarray(obs, dtype=np.float64).reshape(-1)[11])

def body_sideslip_rad(obs):                                       # угол скольжения beta из связанных скоростей
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)
    v = max(true_airspeed_ft_s(obs), 1.0)                         # защита от деления на ~0 при V≈0
    return float(math.asin(np.clip(obs[1] / v, -1.0, 1.0)))       # beta = asin(v/V), с клипом

def engine_mu_from_info(info):                                    # вытаскиваем per-engine mu из env info
    damage_state = info.get('damage_state', {}) if isinstance(info, dict) else {}
    em = damage_state.get('engines_mu', {}) if isinstance(damage_state, dict) else {}
    return {eid: float(em.get(eid, em.get(str(eid), 1.0))) for eid in (1,2,3,4)}  # по умолчанию 1.0 (здоров)

def engine_loss_from_info(info, fallback=0.0):                    # переводим per-engine mu в скалярную потерю в [0,1]
    em = engine_mu_from_info(info)
    if not em:
        return float(fallback)                                    # info без engines_mu: оставляем предыдущую оценку
    return float(np.clip(1.0 - em.get(1, 1.0), 0.0, 1.0))         # loss = 1 - mu двигателя №1 (отказывающего)

def per_engine_thrust_lb(obs, throttle, em):                      # вычисляем тягу по двигателю при заданных РУД и mu
    h = altitude_ft(obs)                                          # высота нужна для ISA-Mach
    mach = true_airspeed_ft_s(obs) / isa_speed_of_sound_ft_s(h)   # число Маха = TAS / a(h)
    cluster = engine_model.installed_thrust(mach=mach, altitude_ft=h, throttle=throttle)  # суммарная тяга связки
    per = cluster / 4.0                                           # делим поровну на 4 двигателя
    return np.array([per * float(em.get(eid, 1.0)) for eid in (1,2,3,4)], dtype=np.float64)  # домножаем на mu

def trim_action_for_loss(loss):                                   # смешиваем здоровый и отказной триминг по loss
    f = float(np.clip(loss, 0.0, 1.0))                            # f в [0,1]
    return (1.0 - f) * healthy_trim_action + f * engine_out_trim_action  # выпуклая комбинация

def clip_physical_action(action):                                 # ограничиваем управления физическими лимитами
    a = np.asarray(action, dtype=np.float64).reshape(4)           # 4-вектор
    return np.array([
        np.clip(a[0], -math.radians(25.0), math.radians(25.0)),   # стабилизатор: ±25°
        np.clip(a[1], -math.radians(20.0), math.radians(20.0)),   # элероны: ±20°
        np.clip(a[2], -math.radians(25.0), math.radians(25.0)),   # руль направления: ±25°
        np.clip(a[3], 0.0, 1.0),                                  # РУД: [0, 1]
    ], dtype=np.float64)

def uftc_state_transform_dyn(obs, ref_t, engine_loss_estimate):   # строим UFTC-state из obs относительно time-varying ref
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)
    loss = float(np.clip(engine_loss_estimate, 0.0, 1.0))         # коэф. смешивания в [0,1]
    beta_target = (1.0 - loss) * ref_t['beta'] + loss * beta_engine_out_rad  # смешиваем номин. beta с beta при отказе
    raw = np.array([                                              # state в форме ошибок в физических единицах
        true_airspeed_ft_s(obs) - ref_t['V'],                     # eV = V_actual - V_ref
        altitude_ft(obs) - ref_t['h'],                            # eh = h_actual - h_ref
        wrap_rad(obs[7] - ref_t['theta']),                        # e_theta wrapped в (-pi,pi]
        wrap_rad(obs[8] - ref_t['psi']),                          # e_psi wrapped
        wrap_rad(obs[6] - ref_t['phi']),                          # e_phi wrapped
        body_sideslip_rad(obs) - beta_target,                     # e_beta = beta_actual - beta_target
        obs[3], obs[4], obs[5],                                   # угловые скорости p, q, r
    ], dtype=np.float64)
    return raw / UFTC_STATE_SCALE                                 # нормализуем поканально

def state_feedback_action_dyn(x_uftc, ref_t, engine_loss_estimate):  # закон envelope-аллокатора («L0»-baseline)
    x = np.asarray(x_uftc, dtype=np.float64).reshape(N_UFTC_STATE)   # x — нормализованный UFTC-state
    v_err, h_err = x[0]*UFTC_STATE_SCALE[0], x[1]*UFTC_STATE_SCALE[1]  # обратная нормализация в ft/s, ft
    theta_err = x[2]*UFTC_STATE_SCALE[2]                              # обратная нормализация ошибки theta
    psi_err = x[3]*UFTC_STATE_SCALE[3]                                # обратная нормализация ошибки psi
    phi_err = x[4]*UFTC_STATE_SCALE[4]                                # обратная нормализация ошибки phi
    beta_err = x[5]*UFTC_STATE_SCALE[5]                               # обратная нормализация ошибки beta
    p, q, r = x[6], x[7], x[8]                                        # rates уже в рад/с
    phi_dot_ref = float(ref_t.get('phi_dot', 0.0))                    # FF по скорости крена
    psi_dot_ref = float(ref_t.get('psi_dot', 0.0))                    # FF по скорости курса
    action = trim_action_for_loss(engine_loss_estimate)               # стартуем с смешанного трима
    action[0] += (STATE_FEEDBACK_GAINS['de_h']*h_err                  # инкремент стабилизатора от h, theta, q
                  + STATE_FEEDBACK_GAINS['de_theta']*theta_err
                  + STATE_FEEDBACK_GAINS['de_q']*q)
    action[1] += (STATE_FEEDBACK_GAINS['da_phi']*phi_err              # инкремент элеронов от phi, p, phi_dot_ref
                  + STATE_FEEDBACK_GAINS['da_p']*p
                  + STATE_FEEDBACK_GAINS['da_phi_dot']*phi_dot_ref)
    action[2] += (STATE_FEEDBACK_GAINS['dr_r']*r                      # инкремент руля направления от r, psi, beta, psi_dot_ref
                  + STATE_FEEDBACK_GAINS['dr_psi']*psi_err
                  + STATE_FEEDBACK_GAINS['dr_beta']*beta_err
                  + STATE_FEEDBACK_GAINS['dr_psi_dot']*psi_dot_ref)
    action[3] += STATE_FEEDBACK_GAINS['throttle_v']*v_err + STATE_FEEDBACK_GAINS['throttle_h']*h_err  # инкремент РУД
    return clip_physical_action(action)                              # финальный физический клип

def compose_action(state_feedback, uftc_action_norm):                # сумма state-feedback и residual UFTC
    residual = UFTC_RESIDUAL_SCALE * np.clip(                        # пересчитываем и клипаем нормализованный выход UFTC
        np.asarray(uftc_action_norm, dtype=np.float64).reshape(4), -1.0, 1.0)
    return clip_physical_action(np.asarray(state_feedback, dtype=np.float64).reshape(4) + residual)


## 5б. Классический PID-baseline для сравнения

PID-baseline отрабатывает то же расписание координированного разворота и использует те же физические лимиты приводов, что и UFTC. Это контроллер с фиксированной структурой и фиксированными коэффициентами: gain-ы, рабочая точка трима и цели задаются один раз до старта и **не** меняются после отказа двигателя. У PID нет residual-управления UFTC, нет адаптивной аллокации, нет L4 D-SAC, нет UUB-монитора и нет переключения трима по оценке потери двигателя.


In [ ]:
from tensoraerospace.agent.pid import PID                       # PID-кирпичик из репозитория

PID_GAINS = {                                                     # коэффициенты PID-baseline
    # PID.select_action использует error = setpoint - measurement.
    # Знаки выбраны под соглашение виртуального управления B-747:
    # отрицательный стабилизатор = нос вверх, положительный элерон/правый крен соответствует знаку phi.
    'de_h':        dict(kp=-2.5e-3, ki=-1.0e-6, kd=0.0),         # стабилизатор по ошибке высоты
    'de_theta':    dict(kp=-1.8,    ki=-2.0e-2, kd=-1.0),         # стабилизатор по ошибке тангажа
    'throttle_v':  dict(kp=+1.2e-1, ki=+2.0e-4, kd=0.0),         # РУД по ошибке скорости
    'throttle_h':  dict(kp=+2.0e-4, ki=+2.0e-7, kd=0.0),         # РУД по ошибке высоты
    'da_phi':      dict(kp=+2.5,    ki=+1.0e-2, kd=+1.5),         # элероны по ошибке крена
    'dr_psi_err':  dict(kp=-1.5,    ki=-4.0e-3, kd=-3.0),         # руль направления по ошибке курса
    'dr_beta':     dict(kp=+0.5,    ki=+2.0e-3, kd=0.0),         # руль направления по ошибке скольжения
}
PID_FEEDFORWARD_GAINS = {                                         # FF-коэффициенты у PID-стороны
    'da_phi_dot': 2.25,                                           # FF элеронов по phi_dot_ref (как у UFTC)
}
PID_INCREMENT_LIMITS = {                                          # лимиты инкрементов по каналам — для безопасности
    'de': math.radians(18.0),
    'da': math.radians(16.0),
    'dr': math.radians(20.0),
    'throttle': 0.35,
}

class B747TurnPIDController:
    '''Классический PID-контроллер разворота, используется только как baseline.'''

    def __init__(self, gains=None, feedforward_gains=None):       # позволяем переопределять gain-ы для подбора
        self.gains = dict(PID_GAINS if gains is None else gains)
        self.feedforward_gains = dict(PID_FEEDFORWARD_GAINS if feedforward_gains is None else feedforward_gains)
        self._make_axes()                                         # создаём по PID на каждый скалярный канал

    def _make_axes(self):                                         # инстанцируем все базовые PID
        self.de_h = PID(env=None, dt=DT, **self.gains['de_h'])
        self.de_theta = PID(env=None, dt=DT, **self.gains['de_theta'])
        self.throttle_v = PID(env=None, dt=DT, **self.gains['throttle_v'])
        self.throttle_h = PID(env=None, dt=DT, **self.gains['throttle_h'])
        self.da_phi = PID(env=None, dt=DT, **self.gains['da_phi'])
        self.dr_psi_err = PID(env=None, dt=DT, **self.gains['dr_psi_err'])
        self.dr_beta = PID(env=None, dt=DT, **self.gains['dr_beta'])
        self.axes = [
            self.de_h, self.de_theta, self.throttle_v, self.throttle_h,
            self.da_phi, self.dr_psi_err, self.dr_beta,
        ]

    def reset(self):                                              # сброс интеграторов/производных по всем осям
        for axis in self.axes:
            axis.reset()

    def predict(self, obs, ref_t):                                # одношаговое управление по obs и заданию
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        beta_target = float(ref_t['beta'])                        # PID всегда держит beta=0 (без смешивания с отказом)

        v_actual = true_airspeed_ft_s(obs)                        # текущая TAS
        h_actual = altitude_ft(obs)                               # текущая высота
        theta_actual = float(obs[7])                              # текущий тангаж
        phi_actual = float(obs[6])                                # текущий крен
        beta_actual = body_sideslip_rad(obs)                      # текущее скольжение
        psi_err = wrap_rad(float(obs[8]) - float(ref_t['psi']))   # ошибка курса с заворотом

        # Фиксированный PID-baseline: без оценки потери двигателя и без перепланирования трима после отказа.
        base = healthy_trim_action.copy()                         # всегда здоровый крейсерский трим
        de_increment = (
            self.de_h.select_action(ref_t['h'], h_actual)         # PID по высоте
            + self.de_theta.select_action(ref_t['theta'], theta_actual)  # PID по тангажу
        )
        da_increment = (
            self.da_phi.select_action(ref_t['phi'], phi_actual)   # PID по крену
            + self.feedforward_gains['da_phi_dot'] * float(ref_t.get('phi_dot', 0.0))  # FF по phi_dot
        )
        dr_increment = (
            self.dr_psi_err.select_action(0.0, psi_err)           # PID по ошибке курса
            + self.dr_beta.select_action(beta_target, beta_actual)  # PID по скольжению
        )
        throttle_increment = (
            self.throttle_v.select_action(ref_t['V'], v_actual)   # PID по скорости
            + self.throttle_h.select_action(ref_t['h'], h_actual) # PID по высоте (малая поправка)
        )

        increments = np.array([                                   # собираем и клипуем поканальные инкременты
            np.clip(de_increment, -PID_INCREMENT_LIMITS['de'], PID_INCREMENT_LIMITS['de']),
            np.clip(da_increment, -PID_INCREMENT_LIMITS['da'], PID_INCREMENT_LIMITS['da']),
            np.clip(dr_increment, -PID_INCREMENT_LIMITS['dr'], PID_INCREMENT_LIMITS['dr']),
            np.clip(throttle_increment, -PID_INCREMENT_LIMITS['throttle'], PID_INCREMENT_LIMITS['throttle']),
        ], dtype=np.float64)
        return clip_physical_action(base + increments)            # финальный клип по физическим лимитам


def make_pid_controller():                                        # фабрика, повторяющая интерфейс make_controller()
    return B747TurnPIDController()


## 6. Предобученные веса L4 D-SAC (опционально)

Если в `artifacts/dsac/b747_engine_out_v1/` лежат `actor.pt` / `critic*.pt` / `target*.pt` — подгружаем их; иначе откатываемся к случайно инициализированному актору с печатью предупреждения.

**Оговорка**: эти веса обучены на задаче с отказом двигателя в стационарном триме, *не* в координированном развороте. На манёвре актёр даст ухудшенное качество — это честный результат и аргумент для расширения curriculum, а не отказ стека.


In [ ]:
from tensoraerospace.agent.uftc.l4.dsac import DSACOuter        # класс L4 D-SAC outer-loop

_REL = Path('artifacts/dsac/b747_engine_out_v1')                  # каноничный относительный путь
_candidates = [_REL]                                              # список кандидатов на случай разных cwd
_cwd = Path.cwd().resolve()                                       # абсолютный cwd
for parent in [_cwd, *_cwd.parents]:                              # идём вверх по дереву в поиске artifacts
    cand = parent / _REL
    if cand not in _candidates:
        _candidates.append(cand)
weights_path = None                                               # финальный путь (None если не нашли)
for cand in _candidates:                                          # берём первого кандидата с actor.pt
    if cand.exists() and (cand / 'actor.pt').exists():
        weights_path = cand
        break
PRETRAINED_AVAILABLE = weights_path is not None                   # булев флаг для дальнейшего кода

def _load_pretrained_into(ctl_target):                            # копируем веса из свежего DSACOuter в ctl.l4
    pretrained = DSACOuter.from_pretrained(weights_path)
    ctl_target.l4.actor.load_state_dict(pretrained.actor.state_dict())     # веса актёра
    ctl_target.l4.critic1.load_state_dict(pretrained.critic1.state_dict()) # критик 1
    ctl_target.l4.critic2.load_state_dict(pretrained.critic2.state_dict()) # критик 2
    ctl_target.l4.target1.load_state_dict(pretrained.target1.state_dict()) # target 1 (Polyak)
    ctl_target.l4.target2.load_state_dict(pretrained.target2.state_dict()) # target 2 (Polyak)

if PRETRAINED_AVAILABLE:                                          # печать статуса
    print(f'Найдены предобученные веса L4 в {weights_path}')
else:
    print('Предобученные веса L4 не найдены — используем случайно инициализированного актёра (демо eval_mode)')


## 7. Рантайм замкнутого цикла (rollout)

`run_uftc_episode_turn(damage_profile, ctl)` повторяет схему Phase 4, но скармливает time-varying задание в `uftc_state_transform_dyn` и `state_feedback_action_dyn`. Аргумент `reference` у UFTC остаётся нулевым; зависящее от времени задание попадает только через уставку envelope-аллокатора.


In [ ]:
def run_uftc_episode_turn(damage_profile, *, ctl):
    env = make_env(damage_profile=damage_profile)                 # свежая среда на каждый прогон
    obs, _ = env.reset()                                          # начальное observation (в трим-режиме)
    ctl.reset()                                                   # обнуляем внутреннее состояние контроллера
    engine_loss_estimate = 0.0                                    # скалярная оценка потери двигателя от FDD

    log_keys = [                                                  # полный список ключей, логируемых на каждом шаге
        'V', 'h', 'theta', 'psi', 'phi', 'beta',                  # ошибки слежения (actual - reference)
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',  # сырые измерения
        'x_e', 'y_e',                                             # ground-track в NED, ft
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',  # задание на момент логирования
        'p', 'q', 'r',                                            # угловые скорости (deg/s для графиков)
        'de', 'da', 'dr', 'throttle',                             # выданные команды на привода
        'engine_loss_estimate',                                   # severity FDD в [0,1]
        'T1','T2','T3','T4','T_total',                            # тяга по двигателям и суммарная
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',  # диагностика FDD
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',             # диагностика L4 D-SAC
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd', # компоненты композитного Ляпунова
        'alarm_level', 'mu_uub_pred',                             # уровень тревоги и прогноз mu
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}  # преаллоцируем буферы
    t_axis = np.arange(N_EP, dtype=np.float64) * DT               # глобальная ось времени
    t_log_axis = np.zeros(N_EP, dtype=np.float64)                 # фактическое время логирования (post-step)
    macro_events = []                                             # лог сработавших макро-действий
    alarm_trans = []                                              # лог переходов уровня тревоги
    prev_alarm = 'OK'                                             # стартовый уровень тревоги

    for k in range(N_EP - 2):                                     # основной цикл шагов, оставляем запас 2 шага
        t_now = t_axis[k]                                         # текущее sim-время
        ref_t = reference_schedule(float(t_now))                  # задание на t_now
        x_uftc = uftc_state_transform_dyn(obs, ref_t, engine_loss_estimate)  # нормализованный UFTC-state
        feedback_action = state_feedback_action_dyn(x_uftc, ref_t, engine_loss_estimate)  # baseline-управление
        u_norm = ctl.predict(x_uftc, UFTC_REFERENCE, time_step=k) # residual UFTC (нормализованный)
        action = compose_action(feedback_action, u_norm)          # финальная физическая команда

        obs, _, _, trunc, info = env.step(action)                 # шаг среды
        engine_loss_estimate = engine_loss_from_info(info, fallback=engine_loss_estimate)  # обновляем оценку потери
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)  # post-step время для логов
        ref_log = reference_schedule(t_log)                       # задание на post-step моменте
        next_x_uftc = uftc_state_transform_dyn(obs, ref_log, engine_loss_estimate)  # post-step UFTC-state
        ctl.learn(next_x_uftc, UFTC_REFERENCE, time_step=k)       # один шаг online-обучения (RLS / replay / ...)

        diag = ctl.diagnostics()                                  # вытаскиваем диагностику контроллера
        l4_diag = diag.get('l4', {})                              # суб-диагностика L4
        mon_out = getattr(ctl, '_monitor_out', None)              # выход монитора (если включён)
        if mon_out is not None:                                   # извлекаем Ляпуновские компоненты
            v_hj = float(mon_out.components.V_hj)
            v_indi = float(mon_out.components.V_indi)
            v_iadp = float(mon_out.components.V_iadp)
            v_dsac = float(mon_out.components.V_dsac)
            v_fdd = float(mon_out.components.V_fdd)
            v_total = float(mon_out.V_total)                      # взвешенная сумма компонент
            mu_pred = float(mon_out.mu_uub_pred)                  # прогнозируемый радиус UUB
            cur_alarm = str(mon_out.alarm)                        # 'OK' / 'WARN' / 'CRITICAL'
            for ma in mon_out.interventions:                      # фиксируем сработавшие макро-действия
                macro_events.append((k, t_axis[k], str(ma.kind), dict(ma.payload)))
            if cur_alarm != prev_alarm:                           # фиксируем переход уровня тревоги
                alarm_trans.append((k, t_axis[k], prev_alarm, cur_alarm))
                prev_alarm = cur_alarm
        else:                                                     # монитор выключен (Phase 1 / PID-эквивалент)
            v_hj = v_indi = v_iadp = v_dsac = v_fdd = 0.0
            v_total = 0.0; mu_pred = 0.0; cur_alarm = 'OK'
        alarm_int = {'OK': 0, 'WARN': 1, 'CRITICAL': 2}.get(cur_alarm, 0)  # числовой уровень тревоги

        v_actual = true_airspeed_ft_s(obs)                        # log-time TAS
        h_actual = altitude_ft(obs)                               # log-time высота
        theta_actual = math.degrees(obs[7])                       # тангаж в градусах
        psi_actual = wrap_deg(math.degrees(obs[8]))               # курс в градусах, wrapped
        phi_actual = math.degrees(obs[6])                         # крен в градусах
        beta_actual = math.degrees(body_sideslip_rad(obs))        # скольжение в градусах
        x_e_actual = float(obs[9])                                # NED-координата север (ft)
        y_e_actual = float(obs[10])                               # NED-координата восток (ft)
        beta_ref_effective_rad = ((1.0 - engine_loss_estimate) * ref_log['beta']     # смесь номин. и отказного
                                  + engine_loss_estimate * beta_engine_out_rad)
        beta_ref_effective_deg = math.degrees(beta_ref_effective_rad)

        # Логируем post-step state на t + dt и сравниваем с заданием на ту же метку.
        t_log_axis[k] = t_log                                     # сохраняем фактический timestamp
        logs['V'][k] = v_actual - ref_log['V']                    # ошибка слежения V
        logs['h'][k] = h_actual - ref_log['h']                    # ошибка слежения h
        logs['theta'][k] = math.degrees(wrap_rad(obs[7] - ref_log['theta']))  # ошибка theta в deg, wrapped
        logs['psi'][k] = math.degrees(wrap_rad(obs[8] - ref_log['psi']))      # ошибка psi в deg, wrapped
        logs['phi'][k] = math.degrees(wrap_rad(obs[6] - ref_log['phi']))      # ошибка phi в deg, wrapped
        logs['beta'][k] = beta_actual - beta_ref_effective_deg                 # ошибка beta по смешанному ref

        logs['V_actual'][k] = v_actual                            # сырые измерения
        logs['h_actual'][k] = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k] = psi_actual
        logs['phi_actual'][k] = phi_actual
        logs['beta_actual'][k] = beta_actual
        logs['x_e'][k] = x_e_actual
        logs['y_e'][k] = y_e_actual
        logs['V_ref'][k] = ref_log['V']                           # задания на момент лога
        logs['h_ref'][k] = ref_log['h']
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])
        logs['psi_ref'][k] = math.degrees(ref_log['psi'])
        logs['phi_ref'][k] = math.degrees(ref_log['phi'])
        logs['beta_ref'][k] = beta_ref_effective_deg
        logs['p'][k] = math.degrees(obs[3])                       # угловые скорости в deg/s
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(action[0])                   # команды приводов в deg
        logs['da'][k] = math.degrees(action[1])
        logs['dr'][k] = math.degrees(action[2])
        logs['throttle'][k] = float(action[3])                    # РУД в [0,1]
        logs['engine_loss_estimate'][k] = engine_loss_estimate
        thr = per_engine_thrust_lb(obs, float(action[3]), engine_mu_from_info(info))  # тяга по двигателям
        logs['T1'][k], logs['T2'][k], logs['T3'][k], logs['T4'][k] = thr
        logs['T_total'][k] = float(np.sum(thr))                   # суммарная тяга
        logs['severity'][k] = float(diag['severity'])             # severity FDD
        logs['fault_present'][k] = float(diag['fault_present'])   # бинарный флаг fault FDD
        logs['rls_gamma'][k] = float(diag['rls_gamma'])           # адаптивный forgetting gamma RLS
        logs['innovation_norm'][k] = float(diag['innovation_norm'])  # норма innovation FDD
        logs['l4_beta'][k] = float(l4_diag.get('beta_t', 0.0))    # энтропийный вес L4
        logs['l4_replay_size'][k] = float(l4_diag.get('replay_size', 0))  # размер replay-buffer
        last_r_eff = getattr(ctl, '_last_r_eff', None)            # эффективная награда L4 (если есть)
        if last_r_eff is not None:
            logs['l4_r_eff_norm'][k] = float(np.linalg.norm(last_r_eff))
        logs['V_total'][k] = v_total                              # полный композитный Ляпунов
        logs['V_hj'][k] = v_hj                                    # компонент Гамильтона–Якоби
        logs['V_indi'][k] = v_indi                                # компонент AA-INDI
        logs['V_iadp'][k] = v_iadp                                # компонент IADP
        logs['V_dsac'][k] = v_dsac                                # компонент D-SAC
        logs['V_fdd'][k] = v_fdd                                  # компонент FDD
        logs['alarm_level'][k] = float(alarm_int)
        logs['mu_uub_pred'][k] = mu_pred
        if trunc:                                                 # уважаем досрочное завершение, если env вернул trunc
            break

    n = k + 1                                                     # число валидных семплов
    out = {key: vals[:n] for key, vals in logs.items()}           # обрезаем логи до n
    out['t'] = t_log_axis[:n]                                     # подкладываем вектор времени
    out['macro_events'] = macro_events                            # лог срабатываний макро-действий
    out['alarm_transitions'] = alarm_trans                        # лог переходов уровня тревоги
    return out


In [ ]:
def run_pid_episode_turn(damage_profile, *, pid_ctl):
    env = make_env(damage_profile=damage_profile)                 # свежая среда, возможно с расписанием отказа
    obs, _ = env.reset()                                          # стартуем в триме
    pid_ctl.reset()                                               # обнуляем интеграторы/производные PID
    engine_loss_estimate = 0.0                                    # храним для совместимости со схемой логов

    log_keys = [                                                  # та же схема логов, что и у UFTC
        'V', 'h', 'theta', 'psi', 'phi', 'beta',
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',
        'x_e', 'y_e',
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',
        'p', 'q', 'r',
        'de', 'da', 'dr', 'throttle',
        'engine_loss_estimate',
        'T1','T2','T3','T4','T_total',
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd',
        'alarm_level', 'mu_uub_pred',
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}  # преаллоцируем
    t_axis = np.arange(N_EP, dtype=np.float64) * DT
    t_log_axis = np.zeros(N_EP, dtype=np.float64)

    for k in range(N_EP - 2):                                     # основной цикл шагов
        t_now = t_axis[k]
        ref_t = reference_schedule(float(t_now))                  # PID получает то же time-varying задание
        action = pid_ctl.predict(obs, ref_t)                      # одношаговая команда PID

        obs, _, _, trunc, info = env.step(action)
        engine_loss_estimate = engine_loss_from_info(info, fallback=engine_loss_estimate)  # только для печати
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)
        ref_log = reference_schedule(t_log)

        v_actual = true_airspeed_ft_s(obs)
        h_actual = altitude_ft(obs)
        theta_actual = math.degrees(obs[7])
        psi_actual = wrap_deg(math.degrees(obs[8]))
        phi_actual = math.degrees(obs[6])
        beta_actual = math.degrees(body_sideslip_rad(obs))
        beta_ref_effective_deg = math.degrees(ref_log['beta'])    # PID не смешивает: эффективное задание = номинальное

        t_log_axis[k] = t_log
        logs['V'][k] = v_actual - ref_log['V']
        logs['h'][k] = h_actual - ref_log['h']
        logs['theta'][k] = math.degrees(wrap_rad(obs[7] - ref_log['theta']))
        logs['psi'][k] = math.degrees(wrap_rad(obs[8] - ref_log['psi']))
        logs['phi'][k] = math.degrees(wrap_rad(obs[6] - ref_log['phi']))
        logs['beta'][k] = beta_actual - beta_ref_effective_deg

        logs['V_actual'][k] = v_actual                            # сырые измерения (одинаковая структура с UFTC)
        logs['h_actual'][k] = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k] = psi_actual
        logs['phi_actual'][k] = phi_actual
        logs['beta_actual'][k] = beta_actual
        logs['x_e'][k] = float(obs[9])
        logs['y_e'][k] = float(obs[10])
        logs['V_ref'][k] = ref_log['V']
        logs['h_ref'][k] = ref_log['h']
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])
        logs['psi_ref'][k] = math.degrees(ref_log['psi'])
        logs['phi_ref'][k] = math.degrees(ref_log['phi'])
        logs['beta_ref'][k] = beta_ref_effective_deg
        logs['p'][k] = math.degrees(obs[3])
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(action[0])
        logs['da'][k] = math.degrees(action[1])
        logs['dr'][k] = math.degrees(action[2])
        logs['throttle'][k] = float(action[3])
        logs['engine_loss_estimate'][k] = engine_loss_estimate
        thr = per_engine_thrust_lb(obs, float(action[3]), engine_mu_from_info(info))
        logs['T1'][k], logs['T2'][k], logs['T3'][k], logs['T4'][k] = thr
        logs['T_total'][k] = float(np.sum(thr))
        logs['fault_present'][k] = float(engine_loss_estimate > 0.0)  # синтетический флаг по оценке потери
        logs['severity'][k] = engine_loss_estimate
        if trunc:
            break

    n = k + 1
    out = {key: vals[:n] for key, vals in logs.items()}
    out['t'] = t_log_axis[:n]
    out['macro_events'] = []                                      # у PID нет макро-действий
    out['alarm_transitions'] = []                                 # у PID нет монитора
    return out


In [ ]:
def run_open_loop_episode_turn(damage_profile):
    # ИСТИННО разомкнутый прогон: самолёт всё время держит здоровый крейсерский
    # трим. БЕЗ обратной связи, БЕЗ пилотного входа из расписания разворота,
    # БЕЗ адаптации UFTC, БЕЗ компенсации потери двигателя. Когда двигатель
    # отказывает в полёте, аппарат просто расходится — моменты рысканья/крена
    # от асимметричной тяги беспрепятственно нарастают. Расписание задания всё
    # же логируется, чтобы в 3D-просмотрщике можно было наложить «что *должен*
    # делать» против «что фактически делает» (то есть почти ничего).
    env = make_env(damage_profile=damage_profile)                 # свежая среда (с тем же расписанием отказа)
    obs, _ = env.reset()                                          # стартуем в триме

    # Постоянный здоровый трим — без манёвра, без обратной связи.
    nominal_action = np.array([                                   # 4-вектор, применяемый весь эпизод
        float(trim_result.elevator_rad),                          # стабилизатор в здоровом триме
        0.0,                                                      # элероны = 0
        0.0,                                                      # руль направления = 0
        float(trim_result.throttle),                              # РУД здорового трима
    ], dtype=np.float64)

    log_keys = [                                                  # повторяем схему UFTC (большинство полей пустые)
        'V', 'h', 'theta', 'psi', 'phi', 'beta',
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',
        'x_e', 'y_e',
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',
        'p', 'q', 'r',
        'de', 'da', 'dr', 'throttle',
        'engine_loss_estimate',
        'T1', 'T2', 'T3', 'T4', 'T_total',
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd',
        'alarm_level', 'mu_uub_pred',
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}
    t_axis = np.arange(N_EP, dtype=np.float64) * DT
    t_log_axis = np.zeros(N_EP, dtype=np.float64)

    for k in range(N_EP - 2):                                     # шагаем без обратной связи
        t_now = t_axis[k]
        obs, _, _, trunc, info = env.step(nominal_action)         # подаём постоянный здоровый трим
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)

        v_actual = true_airspeed_ft_s(obs)                        # log-time TAS
        h_actual = altitude_ft(obs)                               # высота
        theta_actual = math.degrees(obs[7])                       # тангаж в deg
        psi_actual = wrap_deg(math.degrees(obs[8]))               # курс в deg
        phi_actual = math.degrees(obs[6])                         # крен в deg
        beta_actual = math.degrees(body_sideslip_rad(obs))        # скольжение в deg

        ref_log = reference_schedule(t_log)                       # задание пишем только для оверлея 3D
        logs['V_actual'][k]     = v_actual
        logs['h_actual'][k]     = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k]   = psi_actual
        logs['phi_actual'][k]   = phi_actual
        logs['beta_actual'][k]  = beta_actual
        logs['x_e'][k] = float(obs[9])                            # ground-track север
        logs['y_e'][k] = float(obs[10])                           # ground-track восток

        logs['V_ref'][k]     = V_REF_FT_S                         # постоянная крейсерская уставка V
        logs['h_ref'][k]     = ALT_REF_FT                         # постоянная крейсерская уставка h
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])     # реальный theta_ref из расписания
        logs['psi_ref'][k]   = math.degrees(ref_log['psi'])       # реальный psi_ref из расписания
        logs['phi_ref'][k]   = math.degrees(ref_log['phi'])       # реальный phi_ref из расписания
        logs['beta_ref'][k]  = 0.0                                # beta_ref = 0 в open-loop логе

        logs['p'][k] = math.degrees(obs[3])                       # rates в deg/s
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(nominal_action[0])           # эхо постоянной трим-команды
        logs['da'][k] = math.degrees(nominal_action[1])
        logs['dr'][k] = math.degrees(nominal_action[2])
        logs['throttle'][k] = float(nominal_action[3])

        t_log_axis[k] = t_log
        if trunc:
            break

    n_logged = int(np.argmax(t_log_axis == 0)) if t_log_axis[-1] == 0 else N_EP  # последний ненулевой индекс
    n_logged = min(n_logged if n_logged > 0 else N_EP, N_EP)      # на всякий случай клипуем в N_EP
    logs['t'] = t_log_axis[:n_logged]
    for key in log_keys:
        logs[key] = logs[key][:n_logged]                          # консистентно обрезаем все серии
    return logs

log_open_loop = run_open_loop_episode_turn(TURN_ENGINE_FAILURE)   # запускаем open-loop прогон
print(f'Open-loop прогон: {len(log_open_loop["t"])} шагов, '
      f'финальное t = {log_open_loop["t"][-1]:.2f} с')
print(f'  финальная ошибка psi: {log_open_loop["psi_actual"][-1] - log_open_loop["psi_ref"][-1]:+.2f}°')
print(f'  финальная ошибка phi: {log_open_loop["phi_actual"][-1] - log_open_loop["phi_ref"][-1]:+.2f}°')
print(f'  финальная ошибка V:   {log_open_loop["V_actual"][-1] - log_open_loop["V_ref"][-1]:+.2f} ft/s')
print(f'  финальная ошибка h:   {log_open_loop["h_actual"][-1] - log_open_loop["h_ref"][-1]:+.2f} ft')


## 8. Запуск конфигураций PID и UFTC

* **Phase 1**: `enable_l4_outer=False, enable_monitor=False` — L1-щит выключен (заглушка), только L2 + L3 + envelope-аллокатор.
* **Phase 4**: полный стек + предобученный L4 + композитный Ляпуновский монитор с расслабленной калибровкой `d=(80,)*5`.


In [ ]:
# Прогоны на здоровом baseline-е нужны, чтобы отделить номинальную задержку
# слежения от отклонений, вызванных отказом. При детерминированной динамике
# повреждённая и здоровая траектории совпадают до DAMAGE_TIME.
pid_nominal_ctl = make_pid_controller()                          # свежий PID для номинального baseline-а
log_pid_nominal = run_pid_episode_turn(None, pid_ctl=pid_nominal_ctl)  # PID без отказа = baseline
print(f'PID здоровый baseline: {len(log_pid_nominal["t"])} шагов, '
      f'финальное t = {log_pid_nominal["t"][-1]:.2f} с')

pid_ctl = make_pid_controller()                                  # свежий PID для прогона с отказом
log_pid = run_pid_episode_turn(TURN_ENGINE_FAILURE, pid_ctl=pid_ctl)  # PID с отказом двигателя
print(f'PID damaged: {len(log_pid["t"])} шагов, финальное t = {log_pid["t"][-1]:.2f} с')

ctl_p1_nominal = make_controller(enable_l4_outer=False, enable_trim_free=False,    # Phase 1, без отказа
                                 enable_monitor=False)
log_p1_nominal = run_uftc_episode_turn(None, ctl=ctl_p1_nominal)
print(f'Phase 1 здоровый baseline: {len(log_p1_nominal["t"])} шагов, '
      f'финальное t = {log_p1_nominal["t"][-1]:.2f} с')

ctl_p1 = make_controller(enable_l4_outer=False, enable_trim_free=False,            # Phase 1, с отказом
                         enable_monitor=False)
log_p1 = run_uftc_episode_turn(TURN_ENGINE_FAILURE, ctl=ctl_p1)
print(f'Phase 1 damaged: {len(log_p1["t"])} шагов, финальное t = {log_p1["t"][-1]:.2f} с')

ctl_p4_nominal = make_controller(enable_l4_outer=True, enable_trim_free=True,      # Phase 4, без отказа
                                  enable_monitor=True)
if PRETRAINED_AVAILABLE:                                                            # подгружаем веса L4, если есть
    _load_pretrained_into(ctl_p4_nominal)
log_p4_nominal = run_uftc_episode_turn(None, ctl=ctl_p4_nominal)
print(f'Phase 4 здоровый baseline: {len(log_p4_nominal["t"])} шагов, '
      f'финальное t = {log_p4_nominal["t"][-1]:.2f} с')

ctl_p4 = make_controller(enable_l4_outer=True, enable_trim_free=True,              # Phase 4, с отказом
                         enable_monitor=True)
if PRETRAINED_AVAILABLE:
    _load_pretrained_into(ctl_p4)
    print(f'Phase 4: загружены веса L4 из {weights_path}')
else:
    print('Phase 4: случайно инициализированный L4 (нет предобученных весов)')
if ctl_p4.monitor is not None:
    print(f'Phase 4: mu_uub_pred монитора = {ctl_p4.monitor.mu_uub_pred:.4f}')        # печать начального mu
log_p4 = run_uftc_episode_turn(TURN_ENGINE_FAILURE, ctl=ctl_p4)
print(f'Phase 4 damaged: {len(log_p4["t"])} шагов, финальное t = {log_p4["t"][-1]:.2f} с')

def _aligned_len(log, baseline):                                  # минимальная общая длина для поэлементных операций
    return min(len(log['t']), len(baseline['t']))

def common_t(log, baseline):                                      # выровненный вектор времени для графиков
    n = _aligned_len(log, baseline)
    return np.asarray(log['t'][:n], dtype=np.float64)

def operational_target(log, baseline, key):                       # operational reference = траектория здорового baseline
    n = _aligned_len(log, baseline)
    t = np.asarray(log['t'][:n], dtype=np.float64)
    if key == 'V':
        return np.asarray(baseline['V_actual'][:n], dtype=np.float64)
    if key == 'h':
        return np.asarray(baseline['h_actual'][:n], dtype=np.float64)
    if key == 'theta':
        return np.asarray(baseline['theta_actual'][:n], dtype=np.float64)
    if key == 'psi':
        return np.asarray(baseline['psi_actual'][:n], dtype=np.float64)
    if key == 'phi':
        return np.asarray(baseline['phi_actual'][:n], dtype=np.float64)
    if key == 'beta':
        target = np.asarray(baseline['beta_actual'][:n], dtype=np.float64).copy()
        target[t >= DAMAGE_TIME] = np.asarray(log['beta_ref'][:n], dtype=np.float64)[t >= DAMAGE_TIME]
        return target                                             # после отказа: смешанное задание (engine-out)
    raise KeyError(key)

def operational_deviation(log, baseline, key):                    # operational deviation = actual - operational target
    n = _aligned_len(log, baseline)
    target = operational_target(log, baseline, key)
    if key == 'V':
        return np.asarray(log['V_actual'][:n] - target, dtype=np.float64)
    if key == 'h':
        return np.asarray(log['h_actual'][:n] - target, dtype=np.float64)
    if key == 'theta':
        return np.asarray(log['theta_actual'][:n] - target, dtype=np.float64)
    if key == 'psi':
        return np.array([wrap_deg(a - b) for a, b in zip(log['psi_actual'][:n], target)],   # курс нужно wrap'ить
                        dtype=np.float64)
    if key == 'phi':
        return np.asarray(log['phi_actual'][:n] - target, dtype=np.float64)
    if key == 'beta':
        return np.asarray(log['beta_actual'][:n] - target, dtype=np.float64)
    raise KeyError(key)

# Алиас для обратной совместимости (используется ниже в ноутбуке).
fault_deviation = operational_deviation                           # имя сохраняется для дальнейших ячеек


In [ ]:
# Top-down траектория разворота с отметкой момента отказа двигателя.
FT_PER_NM = 6076.12                                              # перевод: 1 морская миля = 6076.12 ft

def _track_nm(log):                                              # конвертируем NED ft в NM относительно старта
    north_nm = (np.asarray(log['x_e'], dtype=np.float64) - float(log['x_e'][0])) / FT_PER_NM
    east_nm = (np.asarray(log['y_e'], dtype=np.float64) - float(log['y_e'][0])) / FT_PER_NM
    return east_nm, north_nm

def _reference_track_nm():                                       # чисто кинематическая reference-траектория
    north_ft = np.zeros_like(_t_grid)                            # аккумуляторы север / восток
    east_ft = np.zeros_like(_t_grid)
    for i in range(1, len(_t_grid)):                             # интегрируем V * (cos psi, sin psi) по midpoint
        psi_mid = 0.5 * (_psi_grid[i - 1] + _psi_grid[i])
        north_ft[i] = north_ft[i - 1] + V_REF_FT_S * math.cos(float(psi_mid)) * DT
        east_ft[i] = east_ft[i - 1] + V_REF_FT_S * math.sin(float(psi_mid)) * DT
    return east_ft / FT_PER_NM, north_ft / FT_PER_NM             # отдаём в NM

fig, ax = plt.subplots(figsize=(9, 7))                           # квадратный график под top-down
ref_east_nm, ref_north_nm = _reference_track_nm()
ax.plot(ref_east_nm, ref_north_nm, color='black', ls='--', lw=1.1, label='кинематическое задание')

base_east_nm, base_north_nm = _track_nm(log_p4_nominal)          # здоровый baseline-трек
ax.plot(base_east_nm, base_north_nm, color='tab:green', lw=1.2, alpha=0.8, label='здоровый baseline')

for log_, color, label in [                                      # накладываем все damaged-прогоны
    (log_p1, 'tab:gray', 'Phase 1 damaged'),
    (log_pid, 'tab:orange', 'PID damaged'),
    (log_p4, 'tab:blue', 'Phase 4 damaged'),
]:
    east_nm, north_nm = _track_nm(log_)
    ax.plot(east_nm, north_nm, color=color, lw=1.4, label=label)

fail_idx = int(np.argmin(np.abs(log_p4['t'] - DAMAGE_TIME)))     # ищем на треке Phase 4 точку, ближайшую к DAMAGE_TIME
fail_east_nm, fail_north_nm = _track_nm(log_p4)
ax.scatter(fail_east_nm[fail_idx], fail_north_nm[fail_idx],      # маркер X в точке отказа
           s=95, marker='X', color='red', edgecolor='white', linewidth=0.9,
           zorder=5, label=f'отказ двигателя, t={DAMAGE_TIME:.0f} с')
ax.annotate(f'отказ двигателя\nt = {DAMAGE_TIME:.0f} с',         # подпись со стрелкой
            xy=(fail_east_nm[fail_idx], fail_north_nm[fail_idx]),
            xytext=(12, -28), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.0),
            color='red', fontsize=9, ha='left', va='top')

east4_nm, north4_nm = _track_nm(log_p4)                          # отмечаем начало и конец трека Phase 4
ax.scatter(east4_nm[0], north4_nm[0], s=42, marker='o', color='green',
           edgecolor='white', linewidth=0.7, zorder=4, label='старт')
ax.scatter(east4_nm[-1], north4_nm[-1], s=48, marker='s', color='tab:purple',
           edgecolor='white', linewidth=0.7, zorder=4, label='конец')

ax.set_aspect('equal', adjustable='box')                         # равные масштабы осей: форма разворота не искажается
ax.set_xlabel('смещение на восток, NM')
ax.set_ylabel('смещение на север, NM')
ax.set_title('Ground-track координированного разворота с отказом двигателя')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=9)
plt.tight_layout(); plt.show()


## 9. Задание vs фактическое — сравнение из шести панелей

Каждая панель: operational-задание (чёрный пунктир), Phase 1 damaged (серый), Phase 4 damaged (синий). Вертикальная красная линия — отказ двигателя на t = 50 с. Самое информативное — панели крена и курса: видно, как обе конфигурации проходят разворот через точку отказа.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 9.5), sharex=True)   # сетка 3x2 с общей осью x

# Для статьи в роли пунктирного задания — достижимая замкнутая здоровая
# траектория, а не сырое кинематическое расписание. Это убирает номинальную
# задержку слежения из визуального сравнения: до DAMAGE_TIME damaged и
# здоровый прогоны совпадают, поэтому в каждой панели они визуально
# полностью накладываются.
plot_panels = [                                                  # по одной записи на панель: (ключ лога, подпись y)
    ('V_actual', 'TAS, ft/s'),
    ('h_actual', 'высота, ft'),
    ('theta_actual', 'тангаж theta, °'),
    ('psi_actual', 'курс psi, °'),
    ('phi_actual', 'крен phi, °'),
    ('beta_actual', 'скольжение beta, °'),
]

def _plot_actual_vs_nominal(ax, key, ylabel):                    # отрисовка одной панели
    n_ref = min(len(log_p4['t']), len(log_p4_nominal['t']))      # выравниваем длину с baseline-ом
    t_ref = log_p4['t'][:n_ref]
    target_key = key.replace('_actual', '')                      # 'V_actual' -> 'V' и т.п.
    y_ref = operational_target(log_p4, log_p4_nominal, target_key)[:n_ref]
    ax.plot(t_ref, y_ref, color='black', ls='--', lw=1.1, label='operational reference')

    for log_, color, label in [                                  # накладываем все damaged-прогоны
        (log_p1, 'tab:gray', 'Phase 1 damaged'),
        (log_pid, 'tab:orange', 'PID damaged'),
        (log_p4, 'tab:blue', 'Phase 4 damaged'),
    ]:
        n = min(len(log_['t']), n_ref)
        ax.plot(log_['t'][:n], log_[key][:n], color=color, lw=1.2, label=label)

    ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5, label='отказ двигателя')   # маркер отказа
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

for ax, (key, ylabel) in zip(axes.flat, plot_panels):            # обходим панели построчно
    _plot_actual_vs_nominal(ax, key, ylabel)

axes[2, 0].set_xlabel('время, с')                                # подпись x только у нижнего ряда
axes[2, 1].set_xlabel('время, с')
axes[0, 0].legend(loc='upper right', ncol=2, fontsize=8)         # одна легенда на первой панели
fig.suptitle('Координированный разворот с отказом двигателя — operational reference vs actual')
plt.tight_layout(); plt.show()


## 9б. Операционные отклонения во времени

Операционные отклонения `Delta(t) = damaged(t) - healthy_baseline(t)` по каждому каналу. Горизонтальная линия 0 означает, что отказ не добавил отклонения от номинального манёвра. Красная вертикальная линия — отказ двигателя на t = 50 с; затенённая область за ней — пост-аварийное окно, по которому считается RMS.


In [ ]:
# Операционные отклонения: фактическая траектория минус operational reference.
# Это убирает номинальную задержку слежения из графика ошибок. До DAMAGE_TIME
# два детерминированных прогона должны совпадать с точностью до округлений.
dev_panels = [                                                   # одна запись на панель: (ключ, подпись, имя)
    ("V",     "Delta V (ft/s)",    "скорость"),
    ("h",     "Delta h (ft)",      "высота"),
    ("theta", "Delta theta (°)",   "тангаж"),
    ("psi",   "Delta psi (°)",     "курс"),
    ("phi",   "Delta phi (°)",     "крен"),
    ("beta",  "Delta beta (°)",    "скольжение"),
]

def _rms(y):                                                     # численно стабильный хелпер RMS
    y = np.asarray(y, dtype=np.float64)
    if y.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean(y ** 2)))

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)     # 3x2 панелей отклонений
for ax, (key, ylabel, _label) in zip(axes.flat, dev_panels):
    t1 = common_t(log_p1, log_p1_nominal)                        # выровненные оси времени
    tpid = common_t(log_pid, log_pid_nominal)
    t4 = common_t(log_p4, log_p4_nominal)
    d1 = fault_deviation(log_p1, log_p1_nominal, key)            # операционные отклонения по контроллерам
    dpid = fault_deviation(log_pid, log_pid_nominal, key)
    d4 = fault_deviation(log_p4, log_p4_nominal, key)
    post_mask_p1 = t1 >= DAMAGE_TIME                             # маска post-аварии для RMS
    post_mask_pid = tpid >= DAMAGE_TIME
    post_mask_p4 = t4 >= DAMAGE_TIME

    ax.plot(t1, d1, color="tab:gray", lw=1.2, label="Phase 1")    # три кривые отклонений
    ax.plot(tpid, dpid, color="tab:orange", lw=1.2, label="PID")
    ax.plot(t4, d4, color="tab:blue", lw=1.2, label="Phase 4")
    ax.axhline(0.0, color="black", ls="--", lw=1.0, alpha=0.4)    # нулевая линия
    ax.axvline(DAMAGE_TIME, color="red", ls="--", alpha=0.6, label="отказ двигателя")
    ax.axvspan(DAMAGE_TIME, TOTAL_TIME, color="red", alpha=0.05)  # затеняем post-аварийное окно

    stacked = np.concatenate([d1, dpid, d4])                      # робастные пределы по y через перцентили 1/99
    lo, hi = np.percentile(stacked, [1.0, 99.0])
    pad = 0.1 * (hi - lo) if hi > lo else max(1e-4, 0.1 * abs(hi))
    ax.set_ylim(lo - pad, hi + pad)

    rms_p1 = _rms(d1[post_mask_p1])                               # RMS пост-аварии по каждому контроллеру
    rms_pid = _rms(dpid[post_mask_pid])
    rms_p4 = _rms(d4[post_mask_p4])
    pre_max_p4 = float(np.max(np.abs(d4[t4 < DAMAGE_TIME]))) if np.any(t4 < DAMAGE_TIME) else float("nan")  # пред-аварийный max
    ax.set_title(f"{ylabel}    P1: {rms_p1:.3f}   PID: {rms_pid:.3f}   P4: {rms_p4:.3f}   pre: {pre_max_p4:.1e}",
                 fontsize=10)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

axes[2, 0].set_xlabel("время (с)")
axes[2, 1].set_xlabel("время (с)")
axes[0, 0].legend(loc="upper right", fontsize=9)
fig.suptitle("Операционное отклонение: actual − operational reference", y=1.00)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 3.6))                         # кумулятивный |Delta psi|
t1 = common_t(log_p1, log_p1_nominal)
tpid = common_t(log_pid, log_pid_nominal)
t4 = common_t(log_p4, log_p4_nominal)
abs_d_psi_p1 = np.abs(fault_deviation(log_p1, log_p1_nominal, "psi"))
abs_d_psi_pid = np.abs(fault_deviation(log_pid, log_pid_nominal, "psi"))
abs_d_psi_p4 = np.abs(fault_deviation(log_p4, log_p4_nominal, "psi"))
cum_p1 = np.cumsum(abs_d_psi_p1) * DT                             # прямоугольный интеграл
cum_pid = np.cumsum(abs_d_psi_pid) * DT
cum_p4 = np.cumsum(abs_d_psi_p4) * DT
ax.plot(t1, cum_p1, color="tab:gray", lw=1.4, label="Phase 1")
ax.plot(tpid, cum_pid, color="tab:orange", lw=1.4, label="PID")
ax.plot(t4, cum_p4, color="tab:blue", lw=1.4, label="Phase 4")
ax.axvline(DAMAGE_TIME, color="red", ls="--", alpha=0.6, label="отказ двигателя")
ax.axvspan(DAMAGE_TIME, TOTAL_TIME, color="red", alpha=0.05)
ax.set_xlabel("время (с)")
ax.set_ylabel(r"$\int_0^t |\Delta\psi(\tau)|\, d\tau$, °$\cdot$с")
ax.set_title("Кумулятивное абсолютное операционное отклонение по курсу")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


## 10. Управляющие поверхности и тяга

Команды бок-о-бок. Внешний L4 у Phase 4 добавляет к команде envelope-аллокатора небольшие residual-задания, поэтому кривые в основном перекрываются с локальными расхождениями там, где L4 / макро-действия наиболее активны.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6.5), sharex=True)   # 2x2 панелей под de, da, dr, РУД
ctrl_panels = [
    ('de', 'стабилизатор de, °'),
    ('da', 'элероны da, °'),
    ('dr', 'руль направления dr, °'),
    ('throttle', 'РУД [0,1]'),
]
for ax, (key, ylabel) in zip(axes.flat, ctrl_panels):            # перебираем панели
    ax.plot(log_p1['t'], log_p1[key], color='tab:gray', lw=1.0, label='Phase 1')
    ax.plot(log_pid['t'], log_pid[key], color='tab:orange', lw=1.0, label='PID')
    ax.plot(log_p4['t'], log_p4[key], color='tab:blue', lw=1.0, label='Phase 4')
    ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
    ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
axes[1,0].set_xlabel('время, с'); axes[1,1].set_xlabel('время, с')
axes[0,0].legend(loc='upper right', fontsize=9)
fig.suptitle('Управляющие поверхности и РУД — PID baseline vs UFTC')
plt.tight_layout(); plt.show()


## 11. Композитный Ляпуновский монитор — V_total и таймлайн тревоги (Phase 4)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True,    # верх: V_total, низ: уровень тревоги
                         gridspec_kw={'height_ratios': [3, 1]})
mu_pred = float(ctl_p4.monitor.mu_uub_pred)                       # прогнозируемый UUB-радиус из монитора
warn_th = ctl_p4.cfg.monitor_alarm_warn_frac * mu_pred             # абсолютный порог WARN
crit_th = ctl_p4.cfg.monitor_alarm_critical_frac * mu_pred         # абсолютный порог CRITICAL
axes[0].plot(log_p4['t'], log_p4['V_total'], color='tab:blue', lw=1.3, label=r'$V_{\mathrm{total}}(t)$')
axes[0].axhline(mu_pred, color='red', lw=1.3, label=fr'$\mu_{{UUB}}={mu_pred:.3f}$')
axes[0].axhline(crit_th, color='red', lw=1.0, ls='--',
                label=f'CRITICAL = {ctl_p4.cfg.monitor_alarm_critical_frac:.2f} mu')
axes[0].axhline(warn_th, color='darkorange', lw=1.0, ls='--',
                label=f'WARN = {ctl_p4.cfg.monitor_alarm_warn_frac:.2f} mu')
axes[0].axvline(DAMAGE_TIME, color='red', ls=':', alpha=0.55, label='отказ двигателя')
axes[0].set_ylabel(r'$V_{\mathrm{total}}$'); axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left', ncol=2)
axes[1].step(log_p4['t'], log_p4['alarm_level'], color='tab:red', where='post', lw=1.4)  # дискретный уровень тревоги
axes[1].axvline(DAMAGE_TIME, color='red', ls=':', alpha=0.55)
axes[1].set_yticks([0,1,2]); axes[1].set_yticklabels(['OK','WARN','CRITICAL'])
axes[1].set_ylim(-0.2, 2.4)
axes[1].set_xlabel('время, с'); axes[1].set_ylabel('уровень тревоги')
axes[1].grid(True, alpha=0.3)
axes[0].set_title('Монитор Phase 4 — V_total и тревога во время разворота + отказа')
plt.tight_layout(); plt.show()

alarm_int = log_p4['alarm_level']                                 # числовой ряд уровня тревоги
ok_frac   = float(np.mean(alarm_int == 0))                        # доля времени на каждом уровне
warn_frac = float(np.mean(alarm_int == 1))
crit_frac = float(np.mean(alarm_int == 2))
print(f'Диапазон V_total: [{float(np.min(log_p4["V_total"])):.3f}, {float(np.max(log_p4["V_total"])):.3f}], '
      f'среднее = {float(np.mean(log_p4["V_total"])):.3f}')
print(f'Доля времени OK / WARN / CRITICAL: '
      f'{ok_frac*100:.1f}% / {warn_frac*100:.1f}% / {crit_frac*100:.1f}%')
print(f'Переходов тревоги: {len(log_p4["alarm_transitions"])}')
for step_idx, t_, prev, new in log_p4['alarm_transitions'][:20]:  # первые 20 переходов
    print(f'  шаг {step_idx:>5d}, t={t_:>6.2f} с : {prev:>9s} -> {new}')
macro_kinds = Counter(k for _, _, k, _ in log_p4['macro_events'])  # подсчёт сработок макро-действий
print(f'Макро-действий сработало (всего {len(log_p4["macro_events"])}):')
for k_, v_ in sorted(macro_kinds.items(), key=lambda kv: -kv[1]):
    print(f'  {k_:32s} : {v_} раз')


## 12. Тяга по двигателям


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.2))                         # одна панель: все 4 двигателя + сумма
labels = {'T1': 'двигатель 1: левый внешний (отказ)',
          'T2': 'двигатель 2: левый внутренний',
          'T3': 'двигатель 3: правый внутренний',
          'T4': 'двигатель 4: правый внешний'}
for k_, lab in labels.items():                                    # рисуем тягу каждого двигателя
    ax.plot(log_p4['t'], log_p4[k_], label=lab)
ax.plot(log_p4['t'], log_p4['T_total'], color='black', ls='--', lw=1.2, label='суммарная тяга')   # сумма пунктиром
ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.45, label='отказ двигателя')
ax.set_xlabel('время, с'); ax.set_ylabel('тяга, lb')
ax.set_title('Поэлементная тяга во время разворота (Phase 4)')
ax.grid(True, alpha=0.3); ax.legend(loc='upper right', ncol=2)
plt.tight_layout(); plt.show()


## 13. Сравнение RMS, вызванных отказом

Количественная сводка **отклонения actual − operational reference** в пост-аварийном окне `t ∈ [50, 300] с`. Печатаем RMS операционного отклонения по каждому каналу для обеих конфигураций.


In [ ]:
post_t4 = common_t(log_p4, log_p4_nominal) >= DAMAGE_TIME         # пост-аварийные маски по контроллерам
post_t1 = common_t(log_p1, log_p1_nominal) >= DAMAGE_TIME
post_tpid = common_t(log_pid, log_pid_nominal) >= DAMAGE_TIME
pre_t4 = common_t(log_p4, log_p4_nominal) < DAMAGE_TIME           # пред-аварийные маски (для проверки)
pre_t1 = common_t(log_p1, log_p1_nominal) < DAMAGE_TIME
pre_tpid = common_t(log_pid, log_pid_nominal) < DAMAGE_TIME

def rms(y):                                                       # локальный хелпер RMS
    return float(np.sqrt(np.mean(np.asarray(y, dtype=np.float64) ** 2)))

rows = []                                                         # строки таблицы пост-аварийного RMS
pre_rows = []                                                     # строки таблицы пред-аварийного max
for key, label, unit in [
    ('V', 'отклонение скорости', 'ft/s'),
    ('h', 'отклонение высоты', 'ft'),
    ('theta', 'отклонение тангажа', '°'),
    ('psi', 'отклонение курса', '°'),
    ('phi', 'отклонение крена', '°'),
    ('beta', 'отклонение скольжения', '°'),
]:
    d1 = fault_deviation(log_p1, log_p1_nominal, key)             # операционные отклонения по контроллерам
    dpid = fault_deviation(log_pid, log_pid_nominal, key)
    d4 = fault_deviation(log_p4, log_p4_nominal, key)
    r1 = rms(d1[post_t1])                                         # пост-аварийный RMS
    rpid = rms(dpid[post_tpid])
    r4 = rms(d4[post_t4])
    pre1 = float(np.max(np.abs(d1[pre_t1]))) if np.any(pre_t1) else float('nan')      # пред-аварийный |max|
    prepid = float(np.max(np.abs(dpid[pre_tpid]))) if np.any(pre_tpid) else float('nan')
    pre4 = float(np.max(np.abs(d4[pre_t4]))) if np.any(pre_t4) else float('nan')
    rows.append((label, unit, r1, rpid, r4))
    pre_rows.append((label, unit, pre1, prepid, pre4))

print(f'Операционный RMSE (actual − operational reference), t ∈ [{DAMAGE_TIME:.0f}, {TOTAL_TIME:.0f}] с')
print(f'{"канал":<22s}  {"ед.":<6s}  {"Phase 1":>10s}  {"PID":>10s}  {"Phase 4":>10s}  {"P4/PID":>8s}')
print('-' * 79)
for label, unit, r1, rpid, r4 in rows:                            # печатаем строку по каналу
    ratio = r4 / rpid if rpid > 0 else float('nan')
    print(f'{label:<22s}  {unit:<6s}  {r1:>10.4f}  {rpid:>10.4f}  {r4:>10.4f}  {ratio:>8.3f}')

print()
print(f'Пред-аварийный max |actual − operational reference|, t < {DAMAGE_TIME:.0f} с')
print(f'{"канал":<22s}  {"ед.":<6s}  {"Phase 1":>10s}  {"PID":>10s}  {"Phase 4":>10s}')
print('-' * 68)
for label, unit, pre1, prepid, pre4 in pre_rows:                  # проверка детерминизма до отказа
    print(f'{label:<22s}  {unit:<6s}  {pre1:>10.3e}  {prepid:>10.3e}  {pre4:>10.3e}')

# Сводка по финальному состоянию
print()
print('Финальное операционное отклонение (последний семпл):')
for tag, log_, base_ in [
    ('Phase 1', log_p1, log_p1_nominal),
    ('PID', log_pid, log_pid_nominal),
    ('Phase 4', log_p4, log_p4_nominal),
]:
    n = _aligned_len(log_, base_)
    print(f'  {tag}:  Delta V={fault_deviation(log_, base_, "V")[n-1]:+8.3f} ft/s, '
          f'Delta h={fault_deviation(log_, base_, "h")[n-1]:+8.2f} ft, '
          f'Delta phi={fault_deviation(log_, base_, "phi")[n-1]:+7.3f}°, '
          f'Delta psi={fault_deviation(log_, base_, "psi")[n-1]:+7.3f}°')


## 14. 3D WebGL-просмотрщик — open-loop vs PID vs Phase 4

Собираем три самодостаточных WebGL-просмотрщика B-747 для одного и того же манёвра: чистый open-loop, классический PID-baseline и полный стек Phase 4 UFTC. По сгенерированным ссылкам можно открыть просмотрщик в отдельной вкладке.


In [ ]:
import json                                                     # JSON для сериализации flight-лога
from IPython.display import HTML, display                         # для inline-iframe в ноутбуке
from tensoraerospace.visualization.three_d import build_html      # собирает self-contained WebGL HTML-страницу

FT_TO_M = 0.3048                                                  # перевод: 1 ft -> 0.3048 м
UFTC_3D_DIR = Path.cwd() if Path.cwd().name == 'uftc' else Path('example/reinforcement_learning/uftc')   # папка вывода

def _pad_or_trim(values, n):                                      # привести серию к ровно n семплам
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    if arr.size < n:
        arr = np.concatenate([arr, np.full(n - arr.size, arr[-1] if arr.size else 0.0)])  # дополняем последним значением
    elif arr.size > n:
        arr = arr[:n]                                             # обрезаем до длины
    return arr

def build_uftc_b747_3d_flight_log(log, *, damage_time=DAMAGE_TIME):  # формируем JSON-ready словарь для build_html
    t = np.asarray(log['t'], dtype=np.float64)
    n = len(t)
    x_e_m = _pad_or_trim(log['x_e'], n) * FT_TO_M                 # NED север в метрах
    y_e_m = _pad_or_trim(log['y_e'], n) * FT_TO_M                 # NED восток в метрах
    h_m = _pad_or_trim(log['h_actual'], n) * FT_TO_M              # высота в метрах
    attitude_rad = np.column_stack([                              # ориентация (roll, pitch, yaw) в рад
        np.radians(_pad_or_trim(log['phi_actual'], n)),
        np.radians(_pad_or_trim(log['theta_actual'], n)),
        np.radians(_pad_or_trim(log['psi_actual'], n)),
    ])
    params = default_parameters(B747Configuration.NOMINAL)        # масса/инерции для блока metadata
    healthy_state = {                                             # снимок «всё здорово»
        'mu': {'elevator': 1.0, 'aileron': 1.0, 'rudder': 1.0, 'throttle': 1.0},
        'jam': {'elevator': None, 'aileron': None, 'rudder': None, 'throttle': None},
        'tau': {'elevator': 0.0, 'aileron': 0.0, 'rudder': 0.0, 'throttle': 0.0},
        'mu_floor': {'elevator': 0.0, 'aileron': 0.0, 'rudder': 0.0, 'throttle': 0.0},
        'engines_mu': {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0},
        'flap_jam_config': None,
    }
    failed_state = json.loads(json.dumps(healthy_state))          # глубокая копия + ломаем двигатель №1
    failed_state['engines_mu']['1'] = 0.0

    return {                                                      # финальный словарь, потребляемый build_html
        'version': 1,
        'metadata': {
            'model': 'B-747',
            'aircraft_type': 'b747',
            'dt': float(DT),
            'n_steps': int(n),
            'airspeed': float(_pad_or_trim(log['V_actual'], n)[0] * FT_TO_M),    # начальная скорость в м/с
            'split_stab': False,
            'params': {
                'weight_lb': float(params.weight_lb),
                'S_ft2': float(params.S_ft2),
                'b_ft': float(params.b_ft),
                'cbar_ft': float(params.cbar_ft),
                'Ix': float(params.Ix),
                'Iy': float(params.Iy),
                'Iz': float(params.Iz),
                'Ixz': float(params.Ixz),
            },
        },
        'geometry': {'aircraft_type': 'b747', 'sections': []},   # geometry-плейсхолдер (просмотрщик подгружает GLB)
        'trajectory': {                                          # ряды по шагам в SI / радианах
            'time': t.tolist(),
            'position': np.column_stack([x_e_m, y_e_m, -h_m]).tolist(),    # (север, восток, down) в метрах
            'attitude': attitude_rad.tolist(),
            'alpha': np.full(n, float(trim_result.alpha_rad)).tolist(),    # упрощение: constant alpha
            'beta': np.radians(_pad_or_trim(log['beta_actual'], n)).tolist(),
            'wx': np.radians(_pad_or_trim(log['p'], n)).tolist(),
            'wy': np.radians(_pad_or_trim(log['q'], n)).tolist(),
            'wz': np.radians(_pad_or_trim(log['r'], n)).tolist(),
            'stab': np.radians(_pad_or_trim(log['de'], n)).tolist(),
            'ail': np.radians(_pad_or_trim(log['da'], n)).tolist(),
            'dir': np.radians(_pad_or_trim(log['dr'], n)).tolist(),
            'throttle': _pad_or_trim(log['throttle'], n).tolist(),
            'altitude_m': h_m.tolist(),
            'airspeed_mps': (_pad_or_trim(log['V_actual'], n) * FT_TO_M).tolist(),
            'references': {                                       # серии задания для оверлея графиков
                'V': (_pad_or_trim(log['V_ref'], n) * FT_TO_M).tolist(),
                'h': (_pad_or_trim(log['h_ref'], n) * FT_TO_M).tolist(),
                'theta': _pad_or_trim(log['theta_ref'], n).tolist(),
                'roll': _pad_or_trim(log['phi_ref'], n).tolist(),
                'yaw': _pad_or_trim(log['psi_ref'], n).tolist(),
                'beta': _pad_or_trim(log['beta_ref'], n).tolist(),
            },
        },
        'damage_events': [                                        # таймлайн отказов, отображается в просмотрщике
            {
                'time': float(damage_time),
                'label': 'left_outer_engine_flameout_mid_turn',
                'event_type': 'EngineFailureEvent',
                'kind': 'EngineFailureEvent',
                'payload': {'engine_id': 1, 'thrust_fraction': 0.0},
            }
        ],
        'damage_state_history': [                                 # кусочно-постоянное состояние отказов
            {'time': 0.0, 'state': healthy_state},
            {'time': float(damage_time), 'state': failed_state},
        ],
    }

# Собираем сравнимые просмотрщики: open-loop, PID-baseline и Phase 4 UFTC.
UFTC_3D_DIR.mkdir(parents=True, exist_ok=True)                    # создаём папку вывода
viewers = [                                                       # по просмотрщику на конфигурацию
    ('open_loop', log_open_loop, 'B-747 — open-loop (без обратной связи)'),
    ('pid',       log_pid, 'B-747 — PID baseline'),
    ('phase4',    log_p4, 'B-747 — Phase 4 UFTC'),
]
viewer_paths = {}                                                 # пути к сгенерированным HTML
for tag, log, title in viewers:
    path = UFTC_3D_DIR / f'uftc_b747_coordinated_turn_engine_failure_3d_{tag}.html'
    flight_log = build_uftc_b747_3d_flight_log(log)               # собираем JSON-ready словарь
    path.write_text(build_html(flight_log, title=title), encoding='utf-8')   # рендерим полный HTML
    viewer_paths[tag] = path
    print(f'3D-просмотрщик сохранён: {path.resolve()}')

# Адаптивные iframes для сравнения внутри ноутбука; ссылки открывают полный просмотрщик.
display(HTML(
    '<div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(360px, 1fr)); gap:10px;">'
    + ''.join(
        f'<div>'
        f'<div style="font-weight:600; margin-bottom:4px;">{title}</div>'
        f'<a href="{viewer_paths[tag].name}" target="_blank">Открыть в новой вкладке</a>'
        f'<iframe src="{viewer_paths[tag].name}" width="100%" height="620" '
        f'style="border:0; border-radius:6px; background:#111;"></iframe>'
        f'</div>'
        for tag, _, title in viewers
    )
    + '</div>'
))


## 15. Заметки и честные оговорки

* **Точка подачи задания.** Аргумент `reference` у UFTC всё время равен нулю. Манёвр заводится в *уставку envelope-аллокатора* `ref_t`, которую читают `uftc_state_transform_dyn` и `state_feedback_action_dyn`. UFTC видит нетривиальную дрейфующую ошибку, и каскад компенсирует её онлайн. Это «чище» по двум причинам: (а) сохраняется plant-agnostic-интерфейс контроллера (никаких правок в самом контроллере под манёвр); (б) существующий микс трим-режима по отказу (`trim_action_for_loss`) уже живёт в envelope-аллокаторе.
* **PID baseline для сравнения.** Прогон PID использует один и тот же набор gain-ов и фиксированный здоровый трим на весь эпизод. Он не получает оценку потери двигателя, не переключается в трим с отказом и не меняет цели после события. Его задача — служить классическим baseline-ом на переходном процессе отказа.
* **Триггер отказа.** Локальный `DamageProfile` собирается с `EngineFailureEvent(trigger_time=50.0, engine_id=1, thrust_fraction=0.0)` — как пресет `LEFT_OUTER_ENGINE_FAILURE`, но в середине разворота, а не на t=10 с.
* **Несоответствие предобученного L4 актёра.** Актёр в `artifacts/dsac/b747_engine_out_v1/` обучен на сценарии с отказом двигателя в стационарном триме, *не* в координированном развороте. Ожидаем, что Phase 4 будет примерно сопоставим с Phase 1 — возможно, чуть хуже по курсу, потому что reference-коррекция L4 «не знает» про манёвр. Curriculum по нескольким манёврам (горизонталь + разные расписания крена + отказ в случайный t) закрывает этот разрыв; текущее демо показывает, что фреймворк работает end-to-end на нестационарной задаче, а не что L4 оптимален именно здесь.
* **Детерминизм.** `np.random.seed(0)`, `torch.manual_seed(0)`, `AAINDIConfig(seed=0)`, `l4_seed=0`. Прогон воспроизводится бит-в-бит.
* **Аппроксимация координированного разворота.** Угол крена φ задаётся напрямую кусочно-линейным расписанием; ψ_ref(t) — трапециевидный интеграл координированной скорости `g·tan(φ)/V`. Самолёт не обязан выполнять *настоящий* координированный разворот — контроллер адаптируется к любой подаваемой паре (φ_ref, ψ_ref).
